In [ ]:
import pandas as pd

In [ ]:
train = pd.read_csv('train.csv')

In [ ]:
test = pd.read_csv('test.csv')

In [ ]:
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [ ]:
test.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1201,20,RL,71.0,9353,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,Shed,0,7,2006,Oth,Abnorml
1,1202,60,RL,80.0,10400,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,3,2009,WD,Normal
2,1203,50,RM,50.0,6000,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2009,WD,Normal
3,1204,20,RL,75.0,9750,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,10,2009,WD,Normal
4,1205,20,RL,78.0,10140,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,7,2006,WD,Normal


In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1200 non-null   int64  
 1   MSSubClass     1200 non-null   int64  
 2   MSZoning       1200 non-null   object 
 3   LotFrontage    990 non-null    float64
 4   LotArea        1200 non-null   int64  
 5   Street         1200 non-null   object 
 6   Alley          75 non-null     object 
 7   LotShape       1200 non-null   object 
 8   LandContour    1200 non-null   object 
 9   Utilities      1200 non-null   object 
 10  LotConfig      1200 non-null   object 
 11  LandSlope      1200 non-null   object 
 12  Neighborhood   1200 non-null   object 
 13  Condition1     1200 non-null   object 
 14  Condition2     1200 non-null   object 
 15  BldgType       1200 non-null   object 
 16  HouseStyle     1200 non-null   object 
 17  OverallQual    1200 non-null   int64  
 18  OverallC

# First model

In [ ]:
X = train.drop(columns=[
    'SalePrice', 'Id',
    'LotFrontage', 'Alley',
    'MasVnrType', 'MasVnrArea',
    'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2',
    'Electrical', 'FireplaceQu',
    'GarageType', 'GarageYrBlt', 'GarageFinish',
    'GarageQual', 'GarageCond',
    'PoolQC', 'Fence', 'MiscFeature'
])
y = train['SalePrice']


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [ ]:
from sklearn.pipeline import Pipeline

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
from sklearn.compose import make_column_transformer

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
ord_cols = X[[col for col in X.columns if 'Qu' in col]].select_dtypes('object').columns.tolist()
ord_cols += ['ExterCond', 'HeatingQC']


vals = [['Po', 'Fa', 'TA', 'Gd', 'Ex']] * len(ord_cols)


ord_cols += [
    'LotShape', 'Utilities', 'LandSlope',
    'Functional', 'PavedDrive'
]


vals += [
    ['Reg', 'IR1', 'IR2', 'IR3'],
    ['ELO', 'NoSeWa', 'NoSewr', 'AllPub'],
    ['Gtl', 'Mod', 'Sev'],
    ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod',
     'Min2', 'Min1', 'Typ'],
    ['N', 'P', 'Y']
]



In [ ]:
oe = OrdinalEncoder(categories=vals)

In [ ]:
transformer = make_column_transformer((OneHotEncoder(drop='first', handle_unknown='ignore'), [
    'MSSubClass', 'MSZoning', 'Street',
    'LandContour', 'LotConfig', 'Neighborhood',
    'Condition1', 'Condition2',
    'BldgType', 'HouseStyle',
    'RoofStyle', 'RoofMatl',
    'Exterior1st', 'Exterior2nd',
    'Foundation', 'Heating', 'CentralAir',
    'SaleType', 'SaleCondition'
]), (oe, ord_cols),
                         remainder = 'passthrough')

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
pipe = Pipeline([
    ('transform', transformer),
    ('model', LinearRegression())
])

pipe.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('transform',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['MSSubClass', 'MSZoning',
                                                   'Street', 'LandContour',
                                                   'LotConfig', 'Neighborhood',
                                                   'Condition1', 'Condition2',
                                                   'BldgType', 'HouseStyle',
                                                   'RoofStyle', 'RoofMatl',
                                                   'Exterior1st', 'Exterior2nd',
                                                   'Foundation', 'Heatin...
                                                                             ['Po',
                                                                              'Fa',
                                                                              'TA',
                                                                              'Gd',
                                                                              'Ex'],
                                                                             ['Po',
                                                                              'Fa',
                                                                              'TA',
                                                                              'Gd',
                                                                              'Ex'],
                                                                             ['Reg',
                                                                              'IR1',
                                                                              'IR2',
                                                                              'IR3'],
                                                                             ['ELO',
                                                                              'NoSeWa',
                                                                              'NoSewr',
                                                                              'AllPub'],
                                                                             ['Gtl',
                                                                              'Mod',
                                                                              'Sev'],
                                                                             ['Sal',
                                                                              'Sev',
                                                                              'Maj2',
                                                                              'Maj1',
                                                                              'Mod',
                                                                              'Min2',
                                                                              'Min1',
                                                                              'Typ'],
                                                                             ['N',
                                                                              'P',
                                                                              'Y']]),
                                                  ['ExterQual', 'KitchenQual',
                                                   'ExterCond', 'HeatingQC',
                                                   'LotShape', 'Utilities',
                                                   'LandSlope', 'Functional',
                 

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

In [ ]:
np.sqrt(mean_squared_error(y_train, pipe.predict(X_train)))

np.float64(22099.313247659975)

In [ ]:
np.sqrt(mean_squared_error(y_test, pipe.predict(X_test)))

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [7, 12, 17] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


np.float64(36020.08417869178)

In [ ]:
pipe.fit(X, y)

preds = pipe.predict(test[X.columns])

pdf = pd.DataFrame(preds, columns=['SalePrice'])
pdf['Id'] = test.index

pdf.to_csv('preds.csv', index=False)

pdf.head()

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [7, 10, 11, 12, 13, 15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


,SalePrice,Id
0,88083.925755,0
1,199205.354871,1
2,140565.282989,2
3,199906.749741,3
4,129481.734138,4


In [ ]:
**from google.colab import files
**files.download('preds.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Second Model


In [ ]:
train.columns[train.isna().sum() > 0]

Index(['LotFrontage', 'Alley', 'MasVnrType', 'MasVnrArea', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish',
       'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'],
      dtype='object')

In [ ]:
train[train.columns[train.isna().sum() > 0]].isna().sum()/len(train)


,0
LotFrontage,0.175000
Alley,0.937500
MasVnrType,0.597500
MasVnrArea,0.005000
BsmtQual,0.026667
BsmtCond,0.026667
BsmtExposure,0.027500
BsmtFinType1,0.026667
BsmtFinType2,0.027500
FireplaceQu,0.470000


In [ ]:
missing = train[train.columns[train.isna().sum() > 0]]

In [ ]:
nums_missing = missing.select_dtypes(['float', 'int']).columns.tolist()

In [ ]:
cat_missing = missing.select_dtypes('object').columns.tolist()

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
imputer = make_column_transformer((SimpleImputer(), nums_missing),
                                  (SimpleImputer(strategy = "most_frequent"), cat_missing),
                                  remainder = 'passthrough',
                                  verbose_feature_names_out=False)

In [ ]:
imputer.fit_transform(train)

array([[65.0, 196.0, 2003.0, ..., 'WD', 'Normal', 208500],
       [80.0, 0.0, 1976.0, ..., 'WD', 'Normal', 181500],
       [68.0, 162.0, 2001.0, ..., 'WD', 'Normal', 223500],
       ...,
       [65.0, 0.0, 1916.0, ..., 'ConLw', 'Normal', 144000],
       [70.0, 0.0, 2001.0, ..., 'WD', 'Normal', 178000],
       [75.0, 51.0, 1974.0, ..., 'WD', 'Normal', 148000]], dtype=object)

In [ ]:
X = train.drop(columns=[
    'SalePrice', 'Id'])
y = train['SalePrice']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [ ]:
ord_cols = X[
    [col for col in X.columns if 'Qu' in col]
].select_dtypes('object').columns.tolist()

ord_cols += [
    'ExterCond', 'HeatingQC',
    'BsmtCond', 'GarageCond', 'PoolQC'
]

vals = [['Po', 'Fa', 'TA', 'Gd', 'Ex']] * len(ord_cols)

ord_cols += [
    'LotShape', 'Utilities', 'LandSlope',
    'Functional', 'PavedDrive',
    'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'GarageFinish', 'Fence'
]

vals += [
    ['Reg', 'IR1', 'IR2', 'IR3'],
    ['ELO', 'NoSeWa', 'NoSewr', 'AllPub'],
    ['Gtl', 'Mod', 'Sev'],
    ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod',
     'Min2', 'Min1', 'Typ'],
    ['N', 'P', 'Y'],
    ['No', 'Mn', 'Av', 'Gd'],
    ['Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    ['Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    ['Unf', 'RFn', 'Fin'],
    ['MnWw', 'GdWo', 'MnPrv', 'GdPrv']
]

oe = OrdinalEncoder(categories=vals)

In [ ]:
transformer = make_column_transformer((OneHotEncoder(drop='first', handle_unknown='ignore'), [
    'MSSubClass', 'MSZoning', 'Street',
    'LandContour', 'LotConfig', 'Neighborhood',
    'Condition1', 'Condition2',
    'BldgType', 'HouseStyle',
    'RoofStyle', 'RoofMatl',
    'Exterior1st', 'Exterior2nd',
    'Foundation', 'Heating', 'CentralAir',
    'SaleType', 'SaleCondition', 'Alley', 'MasVnrType', 'Electrical',
        'GarageType', 'MiscFeature'
]), (oe, ord_cols),
                         remainder = 'passthrough')



In [ ]:
pipe = Pipeline([ ('impute', imputer),
    ('transform', transformer),
    ('model', LinearRegression())
])

In [ ]:
imputer.set_output(transform='pandas')
pipe.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('impute',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('simpleimputer-1',
                                                  SimpleImputer(),
                                                  ['LotFrontage', 'MasVnrArea',
                                                   'GarageYrBlt']),
                                                 ('simpleimputer-2',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  ['Alley', 'MasVnrType',
                                                   'BsmtQual', 'BsmtCond',
                                                   'BsmtExposure',
                                                   'BsmtFinType1',
                                                   'BsmtFinType2',
                                                   'FireplaceQu', 'GarageType',
                                                   'GarageFinish'...
                                                                              'ALQ',
                                                                              'GLQ'],
                                                                             ['Unf',
                                                                              'RFn',
                                                                              'Fin'],
                                                                             ['MnWw',
                                                                              'GdWo',
                                                                              'MnPrv',
                                                                              'GdPrv']]),
                                                  ['ExterQual', 'BsmtQual',
                                                   'KitchenQual', 'FireplaceQu',
                                                   'GarageQual', 'ExterCond',
                                                   'HeatingQC', 'BsmtCond',
                                                   'GarageCond', 'PoolQC',
                                                   'LotShape', 'Utilities',
                                                   'LandSlope', 'Functional',
                                                   'PavedDrive', 'BsmtExposure',
                                                   'BsmtFinType1',
                                                   'BsmtFinType2',
                                                   'GarageFinish',
                                                   'Fence'])])),
                ('model', LinearRegression())])

In [ ]:
np.sqrt(mean_squared_error(y_train, pipe.predict(X_train)))

np.float64(20560.66924221791)

In [ ]:
np.sqrt(mean_squared_error(y_test, pipe.predict(X_test)))

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [7, 12, 17] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


np.float64(36520.74989341717)

In [ ]:
preds = pipe.predict(test[X.columns])

pdf = pd.DataFrame(preds, columns=['SalePrice'])
pdf['Id'] = test.index

pdf.to_csv('preds.csv', index=False)

pdf.head()

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [7, 10, 11, 12, 13, 15, 17, 21, 23] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


,SalePrice,Id
0,80733.130838,0
1,203484.645943,1
2,129924.875842,2
3,202222.386097,3
4,139862.776498,4


In [ ]:
from google.colab import files
files.download('preds.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>